# Long Short-Term Memory (LSTM)

Companion code for the *Predictive Analytics Lab* (section 21). Run the cell below.

In [ ]:
# One LSTM cell, written out gate by gate, remembering a signal for 50 steps
import numpy as np

sig = lambda z: 1 / (1 + np.exp(-z))
def lstm_step(x, h, c, W, b):
    z = W @ np.r_[h, x] + b
    H = len(h)
    f, i, o, g = sig(z[:H]), sig(z[H:2*H]), sig(z[2*H:3*H]), np.tanh(z[3*H:])
    c = f * c + i * g            # forget part of the old memory, write part of the candidate
    h = o * np.tanh(c)           # output a filtered version of the cell state
    return h, c, f

H, X = 2, 1
rng = np.random.default_rng(0)
W = rng.normal(0, .3, (4 * H, H + X))
W[H:2*H, -1] = 3.0; W[3*H:, -1] = 1.0          # the input gate opens (and the candidate is written) only when x is large
b = np.r_[np.full(H, 4.0), np.full(H, -4.0), np.zeros(2 * H)]   # forget gate ~1 (keep), input gate ~0 (don't overwrite)
h, c = np.zeros(H), np.zeros(H)
for t, x in enumerate([3.0] + [0.0] * 50):     # one input pulse, then 50 empty steps
    h, c, f = lstm_step(np.array([x]), h, c, W, b)
    if t == 0: c0 = c[0]
    if t in (0, 1, 10, 25, 50):
        print(f"t={t:2d}  cell state={c.round(3)}  forget gate={f.round(3)}")
print("After 50 steps the LSTM still holds", round(float(c[0] / c0), 2), "of its stored value;")
print("a vanilla RNN with recurrent factor 0.9 would keep", round(0.9 ** 50, 5), "of the signal.")